In [1]:
import numpy as np
import healpy as hp
import lumos
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, ICRS, get_sun
import astropy.units as u
from astropy.wcs import WCS
from astropy.time import Time
import requests
from skyfield.api import load, wgs84
from lumos.geometry import Surface
from lumos.geometry import Surface
from lumos.brdf.library import BINOMIAL, PHONG
import lumos.conversions
import lumos.brdf.library
import lumos.calculator
import lumos.plot
import numpy as np
import astropy.time
import astropy.coordinates
import astropy.units as u
import satellitemodels
import json

/Users/alexbradshaw/Research/Most Updated Files


In [2]:
#General Functions
# Load ephemeris and timescale
ts = load.timescale()
eph = load('de440s.bsp')
sun = eph['Sun']
earth = eph['Earth']


def healpix_to_lonlat(healpix_id, nside=16):
    theta, phi = hp.pix2ang(nside, healpix_id, nest=False)
    lat = 90 - np.degrees(theta)
    lon = np.degrees(phi)
    lon = ((lon + 180) % 360) - 180
    return lon, lat
    
def horizontal_to_equatorial(latitude, longitude, elevation, azimuth_deg, elevation_obs_deg, julian_day):
    location = EarthLocation(lat=latitude*u.deg, lon=longitude*u.deg, height=elevation*u.m)
    t = Time(julian_day, format='jd')
    altaz = AltAz(az=azimuth_deg*u.deg, alt=elevation_obs_deg*u.deg, location=location, obstime=t)
    eq = altaz.transform_to(ICRS())
    return eq.ra.deg, eq.dec.deg

def sun_altazi(time, ob_location):
    obs_time = astropy.time.Time(time, format='jd')  
    aa_frame = astropy.coordinates.AltAz(obstime=obs_time, location=ob_location)
    sun_altaz = astropy.coordinates.get_sun(obs_time).transform_to(aa_frame)
    
    sun_alt = np.array(sun_altaz.alt.degree)  
    sun_az = np.array(sun_altaz.az.degree)
    
    print("Sun Altitudes:", sun_alt)
    print("Sun Azimuths:", sun_az)
    
    # If it's a single-element array, just return the scalar values
    if sun_alt.size == 1:
        return sun_az.item(), sun_alt.item()
    else:
        return sun_az, sun_alt

In [3]:
#Earth Shadow and Intensity Functions
import requests
import numpy as np
import astropy.coordinates
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, GCRS, get_sun


def satellite_in_earth_shadow(
    jd_time,
    sat_az_deg,
    sat_el_deg,
    slant_range_km,
    obs_lat_deg,
    obs_lon_deg,
    obs_height_m,
    earth_radius_km=6371.0,
):
    """
    Returns True if satellite is in Earth's shadow, False otherwise.
    Uses cylindrical shadow approximation.
    """

    t = Time(jd_time, format="jd")

    observer = EarthLocation(
        lat=obs_lat_deg * u.deg,
        lon=obs_lon_deg * u.deg,
        height=obs_height_m * u.m,
    )

    altaz_frame = AltAz(
        obstime=t,
        location=observer,
    )

    sat_topo = SkyCoord(
        az=sat_az_deg * u.deg,
        alt=sat_el_deg * u.deg,
        distance=slant_range_km * u.km,
        frame=altaz_frame,
    )

    # Satellite vector from Earth center in GCRS
    sat_gcrs = sat_topo.transform_to(GCRS(obstime=t))
    sat_vec = sat_gcrs.cartesian.xyz.to(u.km).value

    # Sun vector from Earth center in GCRS
    sun_gcrs = get_sun(t).transform_to(GCRS(obstime=t))
    sun_vec = sun_gcrs.cartesian.xyz.to(u.km).value

    sun_hat = sun_vec / np.linalg.norm(sun_vec)

    # Satellite must be on anti-sun side
    dot_val = np.dot(sat_vec, sun_hat)

    # Distance from Sun-Earth axis
    perp_dist = np.linalg.norm(np.cross(sat_vec, sun_hat))

    in_shadow = (dot_val < 0.0) and (perp_dist < earth_radius_km)

    return in_shadow


def parameterfunction(jd_time, ra_c, dec_c):

    all_parameters = []

    R_E = 6371.0  # km

    # Observer location
    obs_lat_deg = -29.038169452952143
    obs_lon_deg = 26.402995668269295
    obs_height_m = 1372.0

    south_af = astropy.coordinates.EarthLocation(
        lat=obs_lat_deg * u.deg,
        lon=obs_lon_deg * u.deg,
        height=obs_height_m * u.m
    )

    url = 'https://satchecker.cps.iau.org/fov/satellite-passes/'

    params = {
        'latitude': obs_lat_deg,
        'longitude': obs_lon_deg,
        'elevation': obs_height_m,
        'start_time_jd': jd_time,
        'duration': 60,
        'ra': ra_c,
        'dec': dec_c,
        'fov_radius': 3,
        'group_by': 'satellite',
        'async': False
    }

    try:
        print(f"Requesting JD: {jd_time}")

        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        
        if isinstance(data, list):
            data = data[0]

        satellites = data.get('data', {}).get('satellites', {})

        if not satellites:
            print(f"No satellite data found for Julian date {jd_time}")
            return all_parameters

        for satellite_name, satellite_data in satellites.items():
            print(f"\nProcessing satellite: {satellite_name}")

            for position in satellite_data.get('positions', []):

                observation_time = position['julian_date']
                slant_range = position['range_km']   # km
                azimuth = position['azimuth']        # deg
                altitude = position['altitude']      # deg

                # True orbital altitude above Earth surface
                rho = np.sqrt(
                    R_E**2 +
                    slant_range**2 +
                    2 * R_E * slant_range * np.sin(np.deg2rad(altitude))
                )
                satellite_height = rho - R_E

                # Sun position as seen by observer
                sun_azimuth, sun_altitude = sun_altazi([observation_time], south_af)
                sun_azimuth = np.asarray(sun_azimuth).item()
                sun_altitude = np.asarray(sun_altitude).item()

                # Shadow / illumination check
                in_shadow = satellite_in_earth_shadow(
                    jd_time=observation_time,
                    sat_az_deg=azimuth,
                    sat_el_deg=altitude,
                    slant_range_km=slant_range,
                    obs_lat_deg=obs_lat_deg,
                    obs_lon_deg=obs_lon_deg,
                    obs_height_m=obs_height_m,
                )
                illuminated = not in_shadow

                params_array = np.array([
                    satellite_height,   # km
                    azimuth,            # deg
                    altitude,           # deg
                    sun_azimuth,        # deg
                    sun_altitude        # deg
                ], dtype=float)

                all_parameters.append({
                    'julian_date': observation_time,
                    'satellite': satellite_name,
                    'illuminated': illuminated,
                    'in_shadow': in_shadow,
                    'parameters': params_array
                })

                print(f"  Time: {observation_time:.6f}")
                print(f"  Slant range: {slant_range:.2f} km")
                print(f"  Orbital height: {satellite_height:.2f} km")
                print(f"  Azimuth: {azimuth:.2f}°, Altitude: {altitude:.2f}°")
                print(f"  Sun Azimuth: {sun_azimuth:.2f}°, Sun Altitude: {sun_altitude:.2f}°")
                print(f"  In shadow: {in_shadow}")
                print(f"  Illuminated: {illuminated}")
                print(f"  Parameters array: {params_array}")
                print(f"  Right Ascension: {ra_c}")
                print(f"  Declination: {dec_c}")

    except Exception as e:
        print(f"Error processing Julian date {jd_time}: {e}")

    return all_parameters

In [4]:
#Satellite Parameters by HEALPIX ID
a = parameterfunction(2460602.371181, 102.945969, -67.009868)
b = parameterfunction(2460602.368403, 255.219540, -73.218955)
c = parameterfunction(2460602.364931, 225.496807, -77.634596)
d = parameterfunction(2460602.371875, 107.031540, -63.710333)
e = parameterfunction(2460602.369792, 116.572319, -84.686497)
F = parameterfunction(2460602.369097, 216.227783, -84.810601)

Requesting JD: 2460602.371181

Processing satellite: GLOBALSTAR M056 (25945)
Sun Altitudes: [-47.91604432]
Sun Azimuths: [204.22193462]
  Time: 2460602.371181
  Slant range: 2849.99 km
  Orbital height: 1674.63 km
  Azimuth: 153.26°, Altitude: 26.18°
  Sun Azimuth: 204.22°, Sun Altitude: -47.92°
  In shadow: False
  Illuminated: True
  Parameters array: [1674.62799929  153.25932953   26.17698074  204.22193462  -47.91604432]
  Right Ascension: 102.945969
  Declination: -67.009868
Sun Altitudes: [-47.91753445]
Sun Azimuths: [204.21623248]
  Time: 2460602.371193
  Slant range: 2854.31 km
  Orbital height: 1674.64 km
  Azimuth: 153.20°, Altitude: 26.09°
  Sun Azimuth: 204.22°, Sun Altitude: -47.92°
  In shadow: False
  Illuminated: True
  Parameters array: [1674.63877821  153.19643564   26.09156041  204.21623248  -47.91753445]
  Right Ascension: 102.945969
  Declination: -67.009868
Sun Altitudes: [-47.91902558]
Sun Azimuths: [204.21052492]
  Time: 2460602.371204
  Slant range: 2858.63 km
 

In [5]:
#Intensity Function
def intensity(parameters, surfaces, illuminated=True, earth_brdf=None, diffuse_sphere=False, filter=False):

    if not illuminated:
        return np.inf

    satellite_height = parameters[0] * 1000
    azimuths = parameters[1]
    altitudes = parameters[2]
    sun_azimuth = parameters[3]
    sun_altitude = parameters[4]

    if diffuse_sphere:
        intensities = diffuse_sphere_model.get_intensity(
            0.65,
            satellite_height,
            altitudes,
            azimuths,
            sun_altitude,
            sun_azimuth
        )
    else:
        intensities = lumos.calculator.get_intensity_observer_frame(
            surfaces,
            satellite_height,
            altitudes,
            azimuths,
            sun_altitude,
            sun_azimuth,
            include_sun=True,
            include_earthshine=earth_brdf is not None,
            earth_panel_density=251,
            earth_brdf=earth_brdf
        )

    if filter:
        intensities = scipy.ndimage.gaussian_filter(intensities, 2, mode=('wrap', 'reflect'))

    calculated_magnitudes = lumos.conversions.intensity_to_ab_mag(intensities)

    return calculated_magnitudes

In [6]:
#Surface Models
chassis_area = 3.65
solar_array_area = 22
chassis_normal = np.array([0,0,-1])
solar_array_normal = np.array([0,1,0])

#one_web_mmt9(overall)(data_points= +10,000)

oneweb_mmt_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.21, 1.04, 7.54)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 3.01))
]

oneweb_mmt_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.35, 1.56, 12.48)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 3.84))
]


#starlink 1.0 (krantz2023)(data_points=4724)

star1_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.59, 0.41, 9.86)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 4.62))
]

star1_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.72, 0.39, 16.49)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 1.58))
]



#starlink 1.0 Visorsat (krantz2023)(data_points=4707)

star1v_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.26, 0.86))
]

star1v_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.58)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.17, 0.28, 0.69))
]



#starlink 1.0 Visorsat (krantz2023)(data_points=4707)

star1d_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.21, 0.18, 8.33)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.26, 0.86))
]

star1d_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.21, 0.18, 8.58)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.17, 0.28, 0.69))
]

#starlink 1.5 (krantz2023)(data_points=2539)

star15_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.31, 0.28, 5.91)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.03, 2.81))
]

star15_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.35, 0.26, 7.50)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.16, 0.19, 0.57))
]


#Oneweb (krantz2023)(data_points=4216)

oneweb_surfaces6d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, 0, 1]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, -1, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([1, 0, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([-1, 0, 0]), PHONG(0.12, 0.10, 5.04)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.00, 0.00, 6.23))
]

oneweb_surfaces2d = [
    Surface(1.0, np.array([0, 0, -1]), PHONG(0.15, 0.08, 6.35)),
    Surface(1.0, np.array([0, 1, 0]), PHONG(0.08, 0.04, 8.78))
]


#Fankhauser_paper
#V1.5
#2D

# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to data measured by Scatterworks.
# The script for fitting is "lab_brdf_fits.ipynb"

B = np.array([[3.34, -98.085]])
C = np.array([[-999.999, 867.538, 1000., 1000., -731.248, 618.552, -294.054, 269.248, -144.853, 75.196]])
lab_chassis_brdf = BINOMIAL(B, C, d = 3.0, l1 = -5)

B = np.array([[0.534, -20.409]])
C = np.array([[-527.765, 1000., -676.579, 430.596, -175.806, 57.879]])
lab_solar_array_brdf = BINOMIAL(B, C, d = 3.0, l1 = -3)

#v1.5
v15_SURFACES_LAB_BRDFS_2D = [
    Surface(chassis_area, chassis_normal, lab_chassis_brdf),
    Surface(solar_array_area, solar_array_normal, lab_solar_array_brdf)
]

# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to actual satellite brightness observations.
# The script for fitting is "infer_brdfs.ipynb"

#v1.5
v15_SURFACES_INFER_BRDFS_2D = [
    Surface(1.0, chassis_normal, PHONG(0.34, 0.40, 8.9)),
    Surface(1.0, solar_array_normal, PHONG(0.15, 0.25, 0.26))]
    
    

#Fankhauser_paper

#6D
    
# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to data measured by Scatterworks.
# The script for fitting is "lab_brdf_fits.ipynb"

B = np.array([[3.34, -98.085]])
C = np.array([[-999.999, 867.538, 1000., 1000., -731.248, 618.552, -294.054, 269.248, -144.853, 75.196]])
lab_chassis_brdf = BINOMIAL(B, C, d = 3.0, l1 = -5)

B = np.array([[0.534, -20.409]])
C = np.array([[-527.765, 1000., -676.579, 430.596, -175.806, 57.879]])
lab_solar_array_brdf = BINOMIAL(B, C, d = 3.0, l1 = -3)

#v1.5
v15_SURFACES_LAB_BRDFS_6D = [
    Surface(chassis_area, chassis_normal, lab_chassis_brdf),
    Surface(chassis_area, np.array([0, 0, 1]), lab_chassis_brdf),
    Surface(0.2, np.array([0, -1, 0]), lab_chassis_brdf),
    Surface(0.2, np.array([0, 1, 0]), lab_chassis_brdf),
    Surface(0.1825, np.array([-1, 0, 0]), lab_chassis_brdf),
    Surface(0.1825, np.array([1, 0, 0]), lab_chassis_brdf),
    Surface(solar_array_area, solar_array_normal, lab_solar_array_brdf)
]
#v1.5
# ------------- Model with Lab Measured BRDFs -------------------------------------------------------
# These BRDFs were found by fitting to actual satellite brightness observations.
# The script for fitting is "infer_brdfs.ipynb"
v15_SURFACES_INFER_BRDFS_6D = [
    Surface(1.0, chassis_normal, PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([0,0,1]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([0,1,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([0,-1,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([1,0,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, np.array([-1,0,0]), PHONG(0.32, 0.41, 8.04)),
    Surface(1.0, solar_array_normal, PHONG(0, 0.14, 1.48))
]

#v2.0
surfaces2 = satellitemodels.get_surfaces()

Using interpolated chassis


In [7]:
#Brightness Intensity Calculator
import numpy as np
from collections import defaultdict

with open("starlink_generations.json", "r") as f:
    starlink_data = json.load(f)

# "STARLINK-1907" -> "v1", "v1.5", "v2 mini", etc.
starlink_version_lookup = {
    d["sat_name"]: d.get("generation", "v1")
    for d in starlink_data.get("satellites", [])
}
surfaces_by_version = {

    "v1": {
        "6d": {
            "star1":  star1_surfaces6d,
            "star1v": star1v_surfaces6d,
            "star1d": star1d_surfaces6d,
        },
        "2d": {
            "star1":  star1_surfaces2d,
            "star1v": star1v_surfaces2d,
            "star1d": star1d_surfaces2d,
        },
    },

    "v1.5": {
        "6d": {
            "krantz": star15_surfaces6d,
            "lab":    v15_SURFACES_LAB_BRDFS_6D,
            "infer":  v15_SURFACES_INFER_BRDFS_6D,
        },
        "2d": {
            "krantz": star15_surfaces2d,
            "lab":    v15_SURFACES_LAB_BRDFS_2D,
            "infer":  v15_SURFACES_INFER_BRDFS_2D,
        },
    },

    "v2 mini": {
        "6d": {
            "v2mini": surfaces2,
        },
        "2d": {
            "v2mini": surfaces2,
        },
    },

    "oneweb": {
        "6d": {
            "oneweb":     oneweb_surfaces6d,
            "oneweb_mmt": oneweb_mmt_surfaces6d,
        },
        "2d": {
            "oneweb":     oneweb_surfaces2d,
            "oneweb_mmt": oneweb_mmt_surfaces2d,
        },
    },
}


datasets = {"a": a, "b": b, "c": c, "d": d, "e": e, "F": F}


results = []



for dataset_name, dataset in datasets.items():
    for row in dataset:
        sat_name = row["satellite"].split(" (")[0].strip()
        sat_upper = sat_name.upper()

        # Determine version
        if "STARLINK" in sat_upper:
            version = starlink_version_lookup.get(sat_name, "v1")
        elif "ONEWEB" in sat_upper:
            version = "oneweb"
        else:
            continue

        surf_pack = surfaces_by_version[version]

        # If this point is in shadow, store that and skip intensity calculation
        if row.get("in_shadow", False):
            results.append({
                "satellite": sat_name,
                "version": version,
                "I6D": None,
                "I2D": None,
                "shadow": True,
            })
            continue

        para = np.asarray(row["parameters"]).reshape(5, 1)

        I6 = {k: float(intensity(para, v)) for k, v in surf_pack["6d"].items()}
        I2 = {k: float(intensity(para, v)) for k, v in surf_pack["2d"].items()}

        results.append({
            "satellite": sat_name,
            "version": version,
            "I6D": I6,
            "I2D": I2,
            "shadow": False,
        })



summary = defaultdict(lambda: {
    "version": None,
    "I6D": defaultdict(list),
    "I2D": defaultdict(list),
    "shadow_count": 0,
    "total_count": 0,
})

for r in results:
    sat = r["satellite"]
    summary[sat]["version"] = r["version"]
    summary[sat]["total_count"] += 1

    if r["shadow"]:
        summary[sat]["shadow_count"] += 1
        continue

    for m, v in r["I6D"].items():
        summary[sat]["I6D"][m].append(v)

    for m, v in r["I2D"].items():
        summary[sat]["I2D"][m].append(v)



print("\nINTENSITY SUMMARY (ALL SATELLITES, ALL MODELS)\n")

for sat in sorted(summary):
    data = summary[sat]
    print(f"\n{sat} | version = {data['version']}")

    # If every point for this satellite was shadowed
    if data["shadow_count"] == data["total_count"]:
        print("  In Earth's Shadow")
        continue

    # Print 6D summaries
    for model, vals in data["I6D"].items():
        arr = np.array(vals, dtype=float)

        if arr.size == 0:
            print(f"  6D [{model:10s}] In Earth's Shadow")
        elif arr.size == 1:
            print(f"  6D [{model:10s}] mean={arr.mean():.4f} std=0.0000")
        else:
            print(f"  6D [{model:10s}] mean={arr.mean():.4f} std={arr.std(ddof=1):.4f}")

    # Print 2D summaries
    for model, vals in data["I2D"].items():
        arr = np.array(vals, dtype=float)

        if arr.size == 0:
            print(f"  2D [{model:10s}] In Earth's Shadow")
        elif arr.size == 1:
            print(f"  2D [{model:10s}] mean={arr.mean():.4f} std=0.0000")
        else:
            print(f"  2D [{model:10s}] mean={arr.mean():.4f} std={arr.std(ddof=1):.4f}")


INTENSITY SUMMARY (ALL SATELLITES, ALL MODELS)


ONEWEB-0212 | version = oneweb
  In Earth's Shadow

ONEWEB-0225 | version = oneweb
  In Earth's Shadow

ONEWEB-0594 | version = oneweb
  6D [oneweb    ] mean=8.2669 std=0.0495
  6D [oneweb_mmt] mean=5.7986 std=0.0516
  2D [oneweb    ] mean=8.2603 std=0.0498
  2D [oneweb_mmt] mean=5.0965 std=0.0545

STARLINK-1169 | version = v1
  In Earth's Shadow

STARLINK-1907 | version = v1
  In Earth's Shadow

STARLINK-2055 | version = v1
  In Earth's Shadow

STARLINK-2356 | version = v1
  In Earth's Shadow

STARLINK-30224 | version = v2 mini
  In Earth's Shadow

STARLINK-30369 | version = v2 mini
  In Earth's Shadow

STARLINK-31095 | version = v2 mini
  In Earth's Shadow

STARLINK-31112 | version = v2 mini
  In Earth's Shadow

STARLINK-31803 | version = v2 mini
  In Earth's Shadow

STARLINK-32094 | version = v2 mini
  In Earth's Shadow
